# VoleykoçAI: Alana Özel Benchmark (Hafta 2.2)

Kendi senaryoma (Türkçe voleybol antrenörlüğü) özel, elle yazılmış 40 çoktan seçmeli soruluk bir test seti. Sorular eğitim verisinde yer almıyor, yani gerçek anlamda held-out.

MMLU genel kültür ölçüyordu ve fine-tune orada base ile başabaştı. Bu test modelin **kendi alanını** ölçüyor: base ile fine-tune arasındaki farkın alanda görünüp görünmediğine bakıyoruz.

Puanlama, MMLU ölçümüyle aynı: harf eşleşmesi (olcum.py mantığı). Çoktan seçmeli olduğu için deterministik ve tekrar edilebilir, hakem gerektirmez.

Colab'da çalışır: `Runtime → Change runtime type → T4 GPU`.

## 1) Kurulum

In [ ]:
%pip install -q unsloth
%pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok. Runtime -> T4 GPU seç."
print("GPU:", torch.cuda.get_device_properties(0).name)

## 2) Benchmark'ı yükle

Benchmark HF'de yayımlandıysa oradan, yoksa doğrudan GitHub'dan çekiyorum.

In [ ]:
import json, urllib.request

# Önce HF dataset, olmazsa GitHub ham dosyası
sorular = None
try:
    from datasets import load_dataset
    ds = load_dataset("berkcangumusisik/voleykoc-benchmark", split="train")
    sorular = [dict(r) for r in ds]
    print("HF'den yüklendi.")
except Exception as e:
    print(f"HF'den yüklenemedi ({type(e).__name__}), GitHub'dan çekiyorum.")
    url = ("https://raw.githubusercontent.com/berkcangumusisik/"
           "voleykocai-llm-finetuning/main/data/benchmark/voleykoc_benchmark.jsonl")
    text = urllib.request.urlopen(url).read().decode("utf-8")
    sorular = [json.loads(l) for l in text.splitlines() if l.strip()]

print(f"{len(sorular)} soru")
print(sorular[0]["soru"])

## 3) Puanlama ve prompt (MMLU ile aynı)

olcum.py'nin harf-eşleşme mantığı. Anlamsal benzerlik kademesi Colab sürüm çakışması yapabildiği için opsiyonel.

In [ ]:
HARFLER = ['A', 'B', 'C', 'D', 'E']


def cevap_dogru_mu(dogru_index, verilen_cevap):
    dogru_harf = HARFLER[dogru_index]
    v = verilen_cevap.upper().strip()
    if v == dogru_harf:
        return True
    if len(v) > 1 and v[1] in [" ", ":", ")", "=", "-", "."]:
        return v[0] == dogru_harf
    for ch in v:
        if ch in HARFLER:
            return ch == dogru_harf
    return False


def prompt_kur(soru, secenekler):
    metin = soru + "\n"
    for j, s in enumerate(secenekler):
        metin += HARFLER[j] + ": " + s + "\n"
    return (
        "Sana soru ve seçenekleri veriyorum. sadece hangi seçeneğin sorunun "
        "doğru cevabı olduğunu yaz. Örneğin 'A' veya 'B' gibi. Lütfen herhangi "
        "bir açıklama yapma!\nSoru: " + metin
    )

## 4) Bir modeli ölçen fonksiyon

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048


def modeli_olc(model_adi, etiket):
    print(f"\n=== {etiket}: {model_adi} ===")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_adi, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"

    dogru = 0
    konu_dogru, konu_toplam = {}, {}
    cevaplar = []
    for s in sorular:
        p = prompt_kur(s["soru"], list(s["secenekler"]))
        metin = tokenizer.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False, add_generation_prompt=True,
        )
        girdi = tokenizer(metin, return_tensors="pt").to("cuda")
        with torch.no_grad():
            cikti = model.generate(
                **girdi, max_new_tokens=8, do_sample=False,
                repetition_penalty=1.3,
                eos_token_id=tokenizer.eos_token_id,  # her model kendi eos'unu bilir
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        c = tokenizer.decode(cikti[0][girdi["input_ids"].shape[1]:], skip_special_tokens=True)
        cevaplar.append(c)

        k = s["konu"]
        konu_toplam[k] = konu_toplam.get(k, 0) + 1
        if cevap_dogru_mu(int(s["cevap"]), c):
            dogru += 1
            konu_dogru[k] = konu_dogru.get(k, 0) + 1

    basari = round(dogru / len(sorular) * 100, 2)
    print(f"  {etiket}: {dogru}/{len(sorular)} = %{basari}")
    del model
    torch.cuda.empty_cache()
    return {"etiket": etiket, "model": model_adi, "basari": basari,
            "dogru": dogru, "toplam": len(sorular),
            "konu_dogru": konu_dogru, "konu_toplam": konu_toplam}

## 5) En az 5 modeli ölç

Ödev kriteri: benchmark en az 5 farklı modelde çalıştırılmalı. Base ve fine-tune'un yanına referans olarak 3 açık model daha ekliyorum. Hepsi Unsloth 4-bit, T4'te sığar. Modeller art arda yüklenip ölçülür.

In [ ]:
MODELLER = [
    ("Base Qwen3-4B", "unsloth/Qwen3-4B-Instruct-2507"),
    ("VoleykoçAI (fine-tune)", "berkcangumusisik/voleykoc-qwen3-4b-lora"),
    ("Qwen3-1.7B", "unsloth/Qwen3-1.7B"),
    ("Llama-3.2-3B", "unsloth/Llama-3.2-3B-Instruct"),
    ("Gemma-3-4B", "unsloth/gemma-3-4b-it"),
]

sonuclar = []
for etiket, model_adi in MODELLER:
    try:
        sonuclar.append(modeli_olc(model_adi, etiket))
    except Exception as e:
        print(f"  ! {etiket} atlandı: {type(e).__name__}: {e}")


## 6) Sonuç tablosu ve JSON

In [ ]:
# Liderlik tablosu (tüm modeller, yüksekten düşüğe)
sonuclar_sirali = sorted(sonuclar, key=lambda r: r["basari"], reverse=True)

print("## VoleykoçAI Alan Benchmark: Liderlik Tablosu\n")
print(f"Elle yazılmış {sonuclar[0]['toplam']} çoktan seçmeli voleybol sorusu (held-out).\n")
print("| Model | Başarı |")
print("|---|---:|")
for r in sonuclar_sirali:
    yildiz = "**" if "fine-tune" in r["etiket"] else ""
    print(f"| {yildiz}{r['etiket']}{yildiz} | %{r['basari']} ({r['dogru']}/{r['toplam']}) |")

# Base vs fine-tune farkı (asıl ilgilenilen)
b = next((r for r in sonuclar if r["etiket"].startswith("Base")), None)
f = next((r for r in sonuclar if "fine-tune" in r["etiket"]), None)
if b and f:
    print(f"\nBase vs fine-tune farkı: {round(f['basari']-b['basari'],2):+} puan")


In [ ]:
import json
ozet = {
    "benchmark": "berkcangumusisik/voleykoc-benchmark",
    "soru_sayisi": sonuclar[0]["toplam"],
    "modeller": [
        {"etiket": r["etiket"], "model": r["model"], "basari": r["basari"], "dogru": r["dogru"]}
        for r in sonuclar_sirali
    ],
}
if b and f:
    ozet["base_vs_finetune_fark"] = round(f["basari"] - b["basari"], 2)
with open("domain_benchmark_sonuclari.json", "w", encoding="utf-8") as fh:
    json.dump(ozet, fh, ensure_ascii=False, indent=2)
print("domain_benchmark_sonuclari.json yazıldı. İndirip reports/ altına koy.")
